<a href="https://colab.research.google.com/github/abilashkannanv/AIML/blob/main/TrustCart_Phase2_Fake_Review_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Load the reviews.csv dataset
reviews_df = pd.read_csv('/content/sample_data/reviews.csv')

### Data Inspection
Let's inspect the first few rows, the data types, and descriptive statistics of the dataset.

In [2]:
# Display the first 5 rows of the DataFrame
display(reviews_df.head())

,review_text,seller_id,review_type
0,Once enjoying my crisp cool water. Our orders ...,4,genuine
1,This should say it all: we found a dress we lo...,1,genuine
2,Slow service\nBelow average food \nIll pass,2,genuine
3,I hit the Primanti Brothers Market Square loca...,3,genuine
4,An impressive and thoughtfully designed produc...,5,fake_generated


In [3]:
# Get a concise summary of the DataFrame, including data types and non-null values
display(reviews_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   review_text  6000 non-null   object
 1   seller_id    6000 non-null   int64 
 2   review_type  6000 non-null   object
dtypes: int64(1), object(2)
memory usage: 140.8+ KB


None

In [4]:
# Generate descriptive statistics of the DataFrame
display(reviews_df.describe(include='all'))

,review_text,seller_id,review_type
count,6000,6000.000000,6000
unique,4019,NaN,3
top,This product delivers an exceptional experienc...,NaN,genuine
freq,266,NaN,4000
mean,NaN,3.486500,NaN
std,NaN,1.362153,NaN
min,NaN,1.000000,NaN
25%,NaN,2.000000,NaN
50%,NaN,4.000000,NaN
75%,NaN,5.000000,NaN


### Target Variable Conversion and Identification
Let's examine the unique values in the `review_type` column to understand how to convert it into a binary target variable.

In [5]:
# Display unique values in 'review_type'
print("Unique values in 'review_type':")
display(reviews_df['review_type'].unique())

Unique values in 'review_type':


array(['genuine', 'fake_generated', 'fake_templated'], dtype=object)

In [6]:
# Convert 'review_type' into a binary target variable
# We'll map 'genuine' to 1 and 'fake_generated' / 'paraphrased' to 0.
reviews_df['is_genuine'] = reviews_df['review_type'].apply(lambda x: 1 if x == 'genuine' else 0)

print("Value counts for the new binary target variable 'is_genuine':")
display(reviews_df['is_genuine'].value_counts())

# Display the first few rows with the new target variable
print("First 5 rows with the new 'is_genuine' column:")
display(reviews_df.head())

Value counts for the new binary target variable 'is_genuine':


,count
is_genuine,
1,4000
0,2000


First 5 rows with the new 'is_genuine' column:


,review_text,seller_id,review_type,is_genuine
0,Once enjoying my crisp cool water. Our orders ...,4,genuine,1
1,This should say it all: we found a dress we lo...,1,genuine,1
2,Slow service\nBelow average food \nIll pass,2,genuine,1
3,I hit the Primanti Brothers Market Square loca...,3,genuine,1
4,An impressive and thoughtfully designed produc...,5,fake_generated,0


### Input Features and Target Label Identification

Based on the problem statement and the data at hand:

*   **Input Features (X):** `review_text` and `seller_id`.
*   **Target Label (y):** `is_genuine` (the newly created binary variable).

In [7]:
# Explicitly define input features and target label
X = reviews_df[['review_text', 'seller_id']]
y = reviews_df['is_genuine']

print("Identified Input Features (X):")
display(X.head())

print("Identified Target Label (y):")
display(y.head())

Identified Input Features (X):


,review_text,seller_id
0,Once enjoying my crisp cool water. Our orders ...,4
1,This should say it all: we found a dress we lo...,1
2,Slow service\nBelow average food \nIll pass,2
3,I hit the Primanti Brothers Market Square loca...,3
4,An impressive and thoughtfully designed produc...,5


Identified Target Label (y):


,is_genuine
0,1
1,1
2,1
3,1
4,0


### Task 2: Text Preprocessing

Now, let's clean the `review_text` data by applying several preprocessing steps.

In [8]:
# 1. Convert text to lowercase
reviews_df['cleaned_review_text'] = reviews_df['review_text'].str.lower()
print("Reviews after converting to lowercase:")
display(reviews_df[['review_text', 'cleaned_review_text']].head())

Reviews after converting to lowercase:


,review_text,cleaned_review_text
0,Once enjoying my crisp cool water. Our orders ...,once enjoying my crisp cool water. our orders ...
1,This should say it all: we found a dress we lo...,this should say it all: we found a dress we lo...
2,Slow service\nBelow average food \nIll pass,slow service\nbelow average food \nill pass
3,I hit the Primanti Brothers Market Square loca...,i hit the primanti brothers market square loca...
4,An impressive and thoughtfully designed produc...,an impressive and thoughtfully designed produc...


In [9]:
import re
import string

# 2. Remove punctuation, numbers, and special characters
def remove_punc_num_special(text):
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text) # Remove punctuation
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = re.sub(r'[^\w\s]', '', text) # Remove special characters (keep alphanumeric and whitespace)
    return text

reviews_df['cleaned_review_text'] = reviews_df['cleaned_review_text'].apply(remove_punc_num_special)
print("Reviews after removing punctuation, numbers, and special characters:")
display(reviews_df[['review_text', 'cleaned_review_text']].head())

Reviews after removing punctuation, numbers, and special characters:


,review_text,cleaned_review_text
0,Once enjoying my crisp cool water. Our orders ...,once enjoying my crisp cool water our orders w...
1,This should say it all: we found a dress we lo...,this should say it all we found a dress we lov...
2,Slow service\nBelow average food \nIll pass,slow servicenbelow average food nill pass
3,I hit the Primanti Brothers Market Square loca...,i hit the primanti brothers market square loca...
4,An impressive and thoughtfully designed produc...,an impressive and thoughtfully designed produc...


In [10]:
# 3. Normalize whitespace
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

reviews_df['cleaned_review_text'] = reviews_df['cleaned_review_text'].apply(normalize_whitespace)
print("Reviews after normalizing whitespace:")
display(reviews_df[['review_text', 'cleaned_review_text']].head())

Reviews after normalizing whitespace:


,review_text,cleaned_review_text
0,Once enjoying my crisp cool water. Our orders ...,once enjoying my crisp cool water our orders w...
1,This should say it all: we found a dress we lo...,this should say it all we found a dress we lov...
2,Slow service\nBelow average food \nIll pass,slow servicenbelow average food nill pass
3,I hit the Primanti Brothers Market Square loca...,i hit the primanti brothers market square loca...
4,An impressive and thoughtfully designed produc...,an impressive and thoughtfully designed produc...


### Cleaned Text for Tokenization

The `cleaned_review_text` column now contains the preprocessed text, ready for further steps like tokenization and feature extraction.

In [12]:
# Update the input features (X) with the cleaned text
# To avoid SettingWithCopyWarning, we re-initialize X using the cleaned_review_text
X = reviews_df[['cleaned_review_text', 'seller_id']].copy()

print("Updated Input Features (X) with cleaned review text:")
display(X.head())

Updated Input Features (X) with cleaned review text:


,cleaned_review_text,seller_id
0,once enjoying my crisp cool water our orders w...,4
1,this should say it all we found a dress we lov...,1
2,slow servicenbelow average food nill pass,2
3,i hit the primanti brothers market square loca...,3
4,an impressive and thoughtfully designed produc...,5


### Task 3: Tokenization and Padding

Now we will tokenize the cleaned review text, convert it into numerical sequences, and apply padding.

In [13]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize the Tokenizer
# num_words: the maximum number of words to keep, based on word frequency.
#             Only the most common num_words-1 words will be kept.
# oov_token: a token to replace out-of-vocabulary words during text_to_sequence calls.
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")

# Fit the tokenizer on the cleaned review text
# This builds the word index
tokenizer.fit_on_texts(X['cleaned_review_text'])

print("Tokenizer fitted. Vocabulary size:", len(tokenizer.word_index))
# Display the first 10 word-index mappings
print("\nFirst 10 word-index mappings:")
for word, index in list(tokenizer.word_index.items())[:10]:
    print(f"  {word}: {index}")

Tokenizer fitted. Vocabulary size: 23944

First 10 word-index mappings:
  <OOV>: 1
  the: 2
  and: 3
  a: 4
  i: 5
  to: 6
  was: 7
  of: 8
  it: 9
  in: 10


In [14]:
# Convert text into numerical sequences
sequences = tokenizer.texts_to_sequences(X['cleaned_review_text'])

print("\nOriginal text of first review:\n", X['cleaned_review_text'].iloc[0])
print("\nNumerical sequence of first review:\n", sequences[0])

# Determine the maximum sequence length
max_sequence_len = max(len(s) for s in sequences)
print(f"\nMaximum sequence length: {max_sequence_len}")


Original text of first review:
 once enjoying my crisp cool water our orders were taken promptly i ordered the chicken club and customized it with a side of coleslaw my better half had the fish sandwich with a coleslaw as well the food came like the wind the coleslaw was damn good the best our sandwiches couldnt have been better if jesus himself made them all around it was the bees knees

Numerical sequence of first review:
 [320, 1906, 16, 1344, 551, 441, 50, 645, 26, 722, 1754, 5, 92, 2, 153, 1253, 3, 7785, 9, 14, 4, 235, 8, 804, 16, 111, 342, 24, 2, 232, 156, 14, 4, 804, 33, 106, 2, 31, 121, 39, 2, 4771, 2, 804, 7, 1156, 28, 2, 78, 50, 412, 430, 22, 54, 111, 40, 5408, 2021, 146, 90, 37, 166, 9, 7, 2, 5409, 4318]

Maximum sequence length: 932


In [15]:
# Apply padding to ensure uniform input length for the model
# We'll pad to the maximum sequence length found, or a reasonable fixed length if it's too long.
# For this dataset, let's use the max_sequence_len directly.
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_len, padding='post', truncating='post')

print("\nShape of padded sequences:", padded_sequences.shape)
print("\nPadded sequence of first review (first 20 elements):\n", padded_sequences[0][:20])

# Update X with the padded sequences
X_padded = padded_sequences

print("\n'X_padded' is ready for model input.")


Shape of padded sequences: (6000, 932)

Padded sequence of first review (first 20 elements):
 [ 320 1906   16 1344  551  441   50  645   26  722 1754    5   92    2
  153 1253    3 7785    9   14]

'X_padded' is ready for model input.


### Task 4: Dataset Splitting

We will now split the dataset into training and testing sets, ensuring that reviews from the same seller do not appear in both sets to prevent data leakage. This is achieved using group-based splitting.

In [16]:
from sklearn.model_selection import GroupShuffleSplit

# Define the groups (seller_id) for group-based splitting
groups = reviews_df['seller_id']

# Initialize GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Get the train and test indices
for train_idx, test_idx in gss.split(X_padded, y, groups):
    X_train, X_test = X_padded[train_idx], X_padded[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

# Verify that no seller_id appears in both train and test sets
common_sellers = set(groups_train.unique()) & set(groups_test.unique())
print(f"\nCommon seller_ids between train and test sets: {len(common_sellers)}")
if len(common_sellers) == 0:
    print("Successfully ensured no seller_id appears in both train and test sets.")
else:
    print(f"Warning: {len(common_sellers)} seller_ids found in both sets.")

Shape of X_train: (5100, 932)
Shape of X_test: (900, 932)
Shape of y_train: (5100,)
Shape of y_test: (900,)

Common seller_ids between train and test sets: 0
Successfully ensured no seller_id appears in both train and test sets.


### Explanation of Data Leakage Prevention

**Why is leakage prevention critical in real-world ML systems?**

Data leakage occurs when information from the test dataset 'leaks' into the training dataset. This can happen in various ways, such as:

1.  **Target leakage:** When features that are highly correlated with the target variable, and would not be available at prediction time, are used during training.
2.  **Train-test contamination:** When information from the test set is used (directly or indirectly) to create features or models for the training set.

In this specific task, if we didn't use group-based splitting and allowed reviews from the same `seller_id` to appear in both the training and testing sets, the model could inadvertently learn seller-specific patterns. When evaluated on the test set, it would appear to perform better than it would on truly unseen data from new sellers, because it has already 'seen' some data points from those sellers during training. This would lead to an **overoptimistic evaluation of the model's performance**.

**Criticality in real-world ML systems:**

*   **Misleading Performance Metrics:** Data leakage can lead to highly inflated performance metrics (e.g., accuracy, precision, recall) on the test set, making us believe the model is more robust and accurate than it truly is.
*   **Poor Generalization:** A model trained with data leakage will fail to generalize well to new, unseen data in production environments, leading to poor real-world performance and potentially significant business consequences.
*   **Loss of Trust:** Deploying a model that performs poorly in production after showing excellent results during development can lead to a loss of trust in the ML system and the team behind it.
*   **Incorrect Business Decisions:** Decisions based on a model's misleading performance can be flawed, leading to wasted resources, missed opportunities, or negative impacts.

By using `GroupShuffleSplit` with `seller_id` as the grouping variable, we ensure that the model is evaluated on reviews from sellers it has never encountered during training. This provides a more realistic and reliable estimate of the model's ability to generalize to new, unseen sellers, which is crucial for building robust and trustworthy ML systems.